In [10]:
import pandas as pd
from matplotlib import pyplot as plt
import seaborn as sns

df = pd.read_csv('train.csv')
df1 = df[['id', 'title', 'text', 'label']].copy().dropna()
df1.count()

id       24348
title    24348
text     24348
label    24348
dtype: int64

In [26]:
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.feature_extraction.text import TfidfVectorizer
from xgboost import XGBClassifier
from sklearn.preprocessing import FunctionTransformer

def flatten(x):
    return x.ravel()

flatten_transformer = FunctionTransformer(flatten, validate=False)

trainData = pd.read_csv('train1.csv')
trainData = trainData[['id', 'title', 'text', 'label']].copy().dropna().drop(['id'], axis=1)
testData = pd.read_csv('test.csv')

test_ids = testData['id']

y = trainData['label'].astype(int)
X = trainData.drop('label', axis = 1)
X_test = testData.drop(['id'], axis=1)

numFeatures = X.select_dtypes(include=['int64', 'float64']).columns.tolist()
catFeatures = X.select_dtypes(exclude=['int64', 'float64']).columns.tolist()

numTransformer = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

catTransformer = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore'))
])

textTransformer = Pipeline([
    ('imputer', SimpleImputer(strategy='constant', fill_value='')),
    ('flatten', flatten_transformer),
    ('tfidf', TfidfVectorizer(max_features=5000))
])

preprocessor = ColumnTransformer([
    ('num', numTransformer, numFeatures),
    ('cat', catTransformer, ['title']),
    ('text', textTransformer, ['text'])
])

best_score = 0
best_model = None

params = {
    "classifier__max_depth": [2, 4, 5],
    "classifier__n_estimators": [100, 150, 200],
    "classifier__tree_method": ['approx'],
    "classifier__lambda": [0, 1],
    "classifier__alpha": [0, 1]
}

xgbc = XGBClassifier(seed=0, device='cuda')

pipeline = Pipeline([
    ('preprocessing', preprocessor),
    ('classifier', xgbc)
])

grid = GridSearchCV(pipeline, params, cv=5, scoring='f1', n_jobs=1)
grid.fit(X, y)

mean_score = grid.best_score_

print(f"xgb: {mean_score:.6f}")

if mean_score > best_score:
    best_score = mean_score
    best_model = grid.best_estimator_
    print(f"best_params: {grid.best_params_}")

try:
    best_model.fit(X, y)
    final_preds = best_model.predict(X_test)

    submission = pd.DataFrame({
        'id': test_ids,
        'label': final_preds.astype(int)
    })
    submission.to_csv('submission.csv', index=False)
except ValueError:
    pass

xgb: 0.984194
best_params: {'classifier__alpha': 0, 'classifier__lambda': 0, 'classifier__max_depth': 4, 'classifier__n_estimators': 200, 'classifier__tree_method': 'approx'}


In [1]:
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import FunctionTransformer
from catboost import CatBoostClassifier

def flatten(x):
    return x.ravel()

flatten_transformer = FunctionTransformer(flatten, validate=False)
to_dense = FunctionTransformer(lambda x: x.toarray(), accept_sparse=True)

# Загрузка данных
trainData = pd.read_csv('train1.csv')
trainData = trainData[['id', 'title', 'text', 'label']].copy().dropna().drop(['id'], axis=1)
testData = pd.read_csv('test.csv')
test_ids = testData['id']

y = trainData['label'].astype(int)
X = trainData.drop('label', axis=1)
X_test = testData.drop(['id'], axis=1)

# Разделение признаков
numFeatures = X.select_dtypes(include=['int64', 'float64']).columns.tolist()
catFeatures = X.select_dtypes(exclude=['int64', 'float64']).columns.tolist()

# Преобразователи
numTransformer = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

textTransformer = Pipeline([
    ('imputer', SimpleImputer(strategy='constant', fill_value='')),
    ('flatten', flatten_transformer),
    ('tfidf', TfidfVectorizer(max_features=5000)),
    ('to_dense', to_dense)
])

# Предобработка
preprocessor = ColumnTransformer([
    ('num', numTransformer, numFeatures),
    ('text', textTransformer, ['text'])
], sparse_threshold=0.0)

# Настройка поиска по сетке
params = {
    "classifier__depth": [4, 6],
    "classifier__iterations": [100, 150],
    "classifier__l2_leaf_reg": [1, 3, 5],
    "classifier__learning_rate": [0.05, 0.1]
}

cb_clf = CatBoostClassifier(
    cat_features=None,
    random_seed=0,
    task_type='GPU'  # Использовать GPU
)

pipeline = Pipeline([
    ('preprocessing', preprocessor),
    ('classifier', cb_clf)
])

grid = GridSearchCV(pipeline, params, cv=5, scoring='f1', n_jobs=1)
grid.fit(X, y)

mean_score = grid.best_score_
print(f"catboost: {mean_score:.6f}")

best_score = 0
best_model = None

if mean_score > best_score:
    best_score = mean_score
    best_model = grid.best_estimator_
    print(f"best_params: {grid.best_params_}")

# Предсказание
try:
    best_model.fit(X, y)
    final_preds = best_model.predict(X_test)

    submission = pd.DataFrame({
        'id': test_ids,
        'label': final_preds.astype(int)
    })
    submission.to_csv('submission.csv', index=False)
except ValueError:
    pass

0:	learn: 0.6007582	total: 47.9ms	remaining: 4.74s
1:	learn: 0.5092520	total: 73.4ms	remaining: 3.6s
2:	learn: 0.4403316	total: 98.2ms	remaining: 3.18s
3:	learn: 0.3830366	total: 124ms	remaining: 2.97s
4:	learn: 0.3392772	total: 148ms	remaining: 2.8s
5:	learn: 0.3017900	total: 173ms	remaining: 2.71s
6:	learn: 0.2712178	total: 197ms	remaining: 2.62s
7:	learn: 0.2389375	total: 223ms	remaining: 2.56s
8:	learn: 0.2159732	total: 246ms	remaining: 2.48s
9:	learn: 0.1960398	total: 273ms	remaining: 2.46s
10:	learn: 0.1800216	total: 297ms	remaining: 2.4s
11:	learn: 0.1667111	total: 319ms	remaining: 2.34s
12:	learn: 0.1551721	total: 343ms	remaining: 2.29s
13:	learn: 0.1453859	total: 371ms	remaining: 2.28s
14:	learn: 0.1372568	total: 399ms	remaining: 2.26s
15:	learn: 0.1300718	total: 425ms	remaining: 2.23s
16:	learn: 0.1239294	total: 450ms	remaining: 2.19s
17:	learn: 0.1165556	total: 474ms	remaining: 2.16s
18:	learn: 0.1123340	total: 503ms	remaining: 2.14s
19:	learn: 0.1076560	total: 533ms	remaini